# 🎬 Demo en vivo · Sustentación Corte 3
**Olist: del Data Warehouse al procesamiento distribuido con Hadoop**

## ⚠️ Antes de grabar
1. **Entorno de ejecución → Cambiar tipo de entorno → CPU** (el gratuito sirve).
2. Ejecuta **solo la celda 0** unos **20 minutos antes** de grabar. Instala Hadoop, reconstruye la bodega y corre los 6 jobs. Espera a que diga `LISTO`.
3. Durante el video ejecuta las celdas **1 a 8 en orden**. Todas tardan segundos, menos la 3 (un job en vivo, ≈1 minuto).

> Cada celda trae arriba **qué decir** mientras corre.

## Celda 0 · Preparación (NO se graba)

In [ ]:
%%bash
[ -d /content/DWH ] || git clone -q https://github.com/juanjose1099322-lang/DWH.git /content/DWH
cd /content/DWH && git pull -q
apt-get install -y -qq bsdextrautils > /dev/null 2>&1
bash scripts/corte3_todo.sh > /content/preparacion.log 2>&1
tail -15 /content/preparacion.log

## Celda 1 · El clúster está vivo
🗣️ *«Tenemos un clúster Hadoop 3.3.6 pseudo-distribuido: todos los demonios en una máquina, comunicándose por localhost. NameNode y DataNode forman HDFS, el almacenamiento distribuido. ResourceManager y NodeManager forman YARN, el gestor de recursos.»*

In [ ]:
%%bash
. /etc/profile.d/hadoop.sh
echo "== Demonios Java de Hadoop =="; jps | grep -v Jps
echo; echo "== Capacidad de HDFS =="; hdfs dfsadmin -report 2>/dev/null | head -5
echo; echo "== Nodos de YARN =="; yarn node -list 2>/dev/null

## Celda 2 · Los datos en HDFS
🗣️ *«Subimos a HDFS lo que no cabía en la bodega: 42.687 reseñas en texto libre, 97.811 entregas exportadas desde la bodega y 1.000.163 puntos de geolocalización sin limpiar. Bajamos el bloque a 16 MB para que la geolocalización quede en 4 bloques: cada bloque será un map task en paralelo.»*

In [ ]:
%%bash
. /etc/profile.d/hadoop.sh
hdfs dfs -ls -h /olist/raw
echo; echo "== Bloques de la geolocalización =="
hdfs fsck /olist/raw/geolocalizacion.csv -files -blocks 2>/dev/null | grep -E "blk_|Total blocks|HEALTHY"
echo; echo "== Muestra: una reseña cruda en portugués =="
hdfs dfs -cat /olist/raw/resenas.tsv 2>/dev/null | sed -n '3p' | cut -c1-200

## Celda 3 · El código: mapper y reducer del Job 1
🗣️ *«Usamos Hadoop Streaming: mappers y reducers en Python que leen de la entrada estándar y escriben en la salida estándar. El mapper normaliza el texto, quita palabras vacías pero NO la negación «não», y emite un par término-clase por cada palabra y bigrama. El mismo reducer hace de combiner: suma en el nodo del mapper y reduce el tráfico por la red un 63 %.»*

In [ ]:
!sed -n '16,27p' /content/DWH/mapreduce/job1_mapper.py
print("-" * 60)
!sed -n '25,36p' /content/DWH/mapreduce/job1_reducer.py

## Celda 4 · ▶ Un job MapReduce EN VIVO en YARN (≈1 min)
🗣️ *Mientras corre:* «Este es el comando completo de Hadoop Streaming. Con `-files` enviamos los scripts a cada nodo; `-mapper`, `-combiner` y `-reducer` dicen qué ejecutar en cada fase. YARN le asigna un contenedor al ApplicationMaster del job, y este pide contenedores para los map y reduce tasks. Miren el avance: primero sube el map, luego el reduce.»

🗣️ *Al terminar:* «Revisen los contadores: 446 mil pares salen del mapper, el combiner los deja en 163 mil, y el reducer entrega 7.545 términos. Ese es nuestro modelo de lenguaje.»

In [ ]:
%%bash
. /etc/profile.d/hadoop.sh
cd /content/DWH/mapreduce
hdfs dfs -rm -r -f -skipTrash /olist/demo/job1 >/dev/null 2>&1
time hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
  -D mapreduce.job.name="olist-DEMO-job1-en-vivo" -D mapreduce.job.reduces=2 \
  -files texto_pt.py,job1_mapper.py,job1_reducer.py \
  -mapper "python3 job1_mapper.py" \
  -combiner "python3 job1_reducer.py --combiner" \
  -reducer "python3 job1_reducer.py" \
  -input /olist/raw/resenas.tsv -output /olist/demo/job1 2>&1 \
  | grep -E "Running job|map [0-9]+% reduce|completed successfully|Map input records|Map output records|Combine output records|Reduce output records|ResenasEntrenamiento"
echo; echo "== Salida en HDFS =="; hdfs dfs -ls /olist/demo/job1

## Celda 5 · YARN: todas las aplicaciones ejecutadas
🗣️ *«YARN registra cada job como una aplicación. Aquí están los seis jobs del pipeline y el que acabamos de lanzar, todos en SUCCEEDED. Fíjense en los tiempos: cada job tarda casi un minuto aunque procese poco. Ese costo fijo de arrancar contenedores es clave en la comparación con la bodega.»*

In [ ]:
%%bash
. /etc/profile.d/hadoop.sh
yarn application -list -appStates FINISHED 2>/dev/null | awk 'NR>1{print $1, $2, $6, $7}' | column -t
echo; echo "== Duración de los 6 jobs del pipeline (s) =="; cat /content/DWH/evidencias/tiempos_yarn.tsv | awk '{print $1, $3" s"}' | column -t

## Celda 6 · Resultados leídos directo de HDFS
🗣️ *«Leemos los resultados sin sacarlos de HDFS. Temas: 6.460 reseñas hablan de retraso, y el 56 % de esos pedidos llegó a tiempo según la bodega. Distancia: de 4,4 % de entregas tardías en envíos cortos a 12 % en los de más de 2.000 km. Clasificador: la diagonal son los aciertos, 83,5 % en total.»*

In [ ]:
%%bash
. /etc/profile.d/hadoop.sh
echo "== Job 2 · Temas (tema, reseñas, % tardíos, ★ prom, ★ tarde, ★ a tiempo, % negativas) =="
hdfs dfs -cat /olist/mr/job2b_temas/part-* 2>/dev/null | sort -t$'\t' -k2,2nr | column -t -s$'\t'
echo; echo "== Job 3 · Distancia real (banda, envíos, km, días, % tardíos, ★) =="
hdfs dfs -cat /olist/mr/job3b_distancias/part-* 2>/dev/null | grep -v RUTA | cut -f2-7 | sort | column -t -s$'\t'
echo; echo "== Job 4 · Matriz de confusión (real, predicha, n) =="
hdfs dfs -cat /olist/mr/job4_clasificar/part-* 2>/dev/null | sort | column -t

## Celda 7 · Integración: de HDFS de vuelta a la bodega
🗣️ *«Las salidas no se quedan en Hadoop: `cargar_resultados_dw.py` las carga como tablas mr_* en la misma bodega del Corte 2. Ahora podemos cruzarlas con SQL, por ejemplo, las rutas que tardan más de lo que su distancia justifica. SP → RJ tarda 4,7 días de más.»*

In [ ]:
import sys, duckdb
sys.path.insert(0, "/content/DWH/etl")
con = duckdb.connect("/content/DWH/data/olist_dw.duckdb", read_only=True)
print(con.execute("SELECT table_name FROM information_schema.tables WHERE table_name LIKE 'mr_%' ORDER BY 1").df().to_string(index=False))
con.execute('''
  SELECT estado_vendedor || ' → ' || estado_cliente AS ruta, envios, ROUND(km_prom) AS km,
         dias_prom AS dias_reales, dias_esperados, exceso_dias, pct_tardio
  FROM vw_rutas_enriquecidas WHERE envios >= 100
  ORDER BY exceso_dias DESC LIMIT 5''').df()

## Celda 8 · El tablero con la sección Big Data
🗣️ *«Y esto ve la gerencia: el tablero del Corte 2 con una sección nueva alimentada por MapReduce. Quejas por tema, distancia contra retraso, rutas lentas, palabras que anticipan una mala reseña y el clasificador.»* (Baja hasta la sección **Voz del cliente y distancia real**.)

In [ ]:
from IPython.display import HTML
HTML(open("/content/DWH/dashboard/tablero_fragmento.html", encoding="utf-8").read())